In [0]:
df = spark.table("gizmobox.bronze.v_customers")
display(df)

In [0]:
dbutils.data.summarize(df)

## Remove records with NULL customer_id

In [0]:
df.select("customer_id").count()

In [0]:
df1 = df.dropna(subset=['customer_id'])
df1.select("customer_id").count()

In [0]:
df2 = df1.dropDuplicates(subset=["customer_id"])
df2.select("customer_id").count()

In [0]:
df1.display()

In [0]:
df3 = df1.groupBy("created_timestamp").count().withColumnRenamed("count","duplicates").select("created_timestamp","duplicates")
df3.display()

In [0]:
df3.display()

In [0]:
df4 = df1.dropDuplicates(subset=["created_timestamp"])
df4.select("customer_id").count()

In [0]:
%sql
create or replace temporary view unique_v_customers as
select distinct * from gizmobox.bronze.v_customers

In [0]:
%sql
with cte_max as (
    select max(created_timestamp) as max_created_timestamp, customer_id from unique_v_customers group by customer_id
)
select t.*
from unique_v_customers t inner join cte_max c on t.created_timestamp = c.max_created_timestamp and t.customer_id = c.customer_id

In [0]:
%sql
create or replace table gizmobox.silver.customers as 
with cte_max as (
    select max(created_timestamp) as max_created_timestamp, customer_id from unique_v_customers group by customer_id
)
select cast(t.created_timestamp as timestamp),t.customer_id,t.customer_name,cast(t.date_of_birth as date),t.email,cast(t.member_since as date),t.telephone,t.file_path
from unique_v_customers t inner join cte_max c on t.created_timestamp = c.max_created_timestamp and t.customer_id = c.customer_id

In [0]:
%sql
describe extended gizmobox.silver.customers